# Alternative Approaches

This page compares StateCounter to other Python methods for working with combinatorial spaces. Understanding these alternatives will help you decide when StateCounter is the right tool and appreciate what it provides.

The [Motivation](motivation.ipynb) page explains *why* StateCounter exists. This page focuses on *how* it compares to existing tools.

## The Core Problem

Recall that StateCounter solves a specific problem: **random access to combinatorial spaces with automatic index tracking**.

Given a Cartesian product (or more complex combinatorial structure), we want to:
1. Access any element by its flat index
2. Know the component indices for that element
3. Shuffle, sample, or slice without reimplementing index math

Let's see how different approaches handle this.

## itertools.product

Python's `itertools.product` is the standard way to enumerate Cartesian products:

In [1]:
from itertools import product

# Enumerate all combinations of 2 treatments × 3 replicates
treatments = range(2)
replicates = range(3)

for i, (t, r) in enumerate(product(treatments, replicates)):
    print(f"Sample {i}: treatment={t}, replicate={r}")

Sample 0: treatment=0, replicate=0
Sample 1: treatment=0, replicate=1
Sample 2: treatment=0, replicate=2
Sample 3: treatment=1, replicate=0
Sample 4: treatment=1, replicate=1
Sample 5: treatment=1, replicate=2


**Limitation: No random access.** `itertools.product` returns an iterator. To get sample #4, you must iterate through samples 0-3 first:

In [2]:
from itertools import product, islice

# To get sample #4, we must consume the first 4 elements
treatments = range(2)
replicates = range(3)

# This is O(n) - wasteful for large spaces
sample_4 = next(islice(product(treatments, replicates), 4, 5))
print(f"Sample 4: treatment={sample_4[0]}, replicate={sample_4[1]}")

Sample 4: treatment=1, replicate=1


**Limitation: No state tracking.** With `itertools`, you get tuples but no connection back to the original ranges. If you want to modify just the treatment or replicate independently, you're on your own.

**Limitation: Shuffling requires materialization.** To shuffle an `itertools.product`, you must convert it to a list first:

In [3]:
import random
from itertools import product

# Must materialize the entire iterator to shuffle
all_samples = list(product(range(2), range(3)))
random.seed(42)
random.shuffle(all_samples)

for i, (t, r) in enumerate(all_samples):
    print(f"Sample {i}: treatment={t}, replicate={r}")

Sample 0: treatment=1, replicate=0
Sample 1: treatment=0, replicate=1
Sample 2: treatment=0, replicate=2
Sample 3: treatment=1, replicate=1
Sample 4: treatment=0, replicate=0
Sample 5: treatment=1, replicate=2


For small spaces this is fine, but for large combinatorial spaces (millions of combinations), materializing the entire list may be prohibitive.

### StateCounter Comparison

StateCounter provides O(1) random access and shuffles indices rather than materializing all combinations:

In [4]:
from statecounter import Counter, Manager, product as sc_product, shuffle

with Manager():
    treatment = Counter(num_states=2, name='treatment')
    replicate = Counter(num_states=3, name='replicate')
    samples = sc_product([treatment, replicate])
    
    # O(1) random access to sample #4
    samples.state = 4
    print(f"Sample 4: treatment={treatment.state}, replicate={replicate.state}")
    
    # Shuffle without materializing - only indices are permuted
    shuffled = shuffle(samples, seed=42)
    print("\nShuffled order:")
    for _ in shuffled:
        shuffled.print_states()

Sample 4: treatment=0, replicate=2

Shuffled order:
treatment=1, replicate=1
treatment=1, replicate=0
treatment=0, replicate=1
treatment=0, replicate=2
treatment=0, replicate=0
treatment=1, replicate=2


## itertools.chain

For disjoint unions (what StateCounter calls `stack`), Python offers `itertools.chain`:

In [5]:
from itertools import chain

controls = [("control", i) for i in range(2)]
treatments = [("treatment", i) for i in range(3)]

for i, item in enumerate(chain(controls, treatments)):
    print(f"Sample {i}: {item}")

Sample 0: ('control', 0)
Sample 1: ('control', 1)
Sample 2: ('treatment', 0)
Sample 3: ('treatment', 1)
Sample 4: ('treatment', 2)


**Limitation: No tracking of which source is active.** With `chain`, you lose information about which iterator contributed each element. You have to encode that information manually (as we did with the tuples above).

### StateCounter Comparison

StateCounter's `stack` automatically tracks which counter is active:

In [6]:
from statecounter import Counter, Manager, stack

with Manager():
    control = Counter(num_states=2, name='control')
    treatment = Counter(num_states=3, name='treatment')
    samples = stack([control, treatment])
    
    for _ in samples:
        # Automatically know which counter is active (other is None)
        samples.print_states()

control=0, treatment=None
control=1, treatment=None
control=None, treatment=0
control=None, treatment=1
control=None, treatment=2


## NumPy's unravel_index / ravel_multi_index

NumPy provides functions to convert between flat indices and multi-dimensional indices:

In [7]:
import numpy as np

# Shape: 2 treatments × 3 replicates
shape = (2, 3)

# Convert flat index 4 to multi-dimensional indices
# Note: numpy uses row-major (C) order by default
indices = np.unravel_index(4, shape)
print(f"Flat index 4 -> treatment={indices[0]}, replicate={indices[1]}")

# Convert back to flat index
flat = np.ravel_multi_index(indices, shape)
print(f"treatment={indices[0]}, replicate={indices[1]} -> flat index {flat}")

Flat index 4 -> treatment=1, replicate=1
treatment=1, replicate=1 -> flat index 4


This provides random access, which is a key capability! However, there are limitations:

**Limitation: Only works for simple products.** NumPy's functions assume a regular multi-dimensional array shape. They can't handle:
- Disjoint unions (stack)
- Slicing or reversing
- Nested combinations of operations

In [8]:
import numpy as np

# What if we want: 2 controls + (2 treatments × 3 replicates)?
# That's 2 + 6 = 8 total, but NOT a rectangular array.
# NumPy can't represent this directly.

# We'd need manual logic:
def get_indices_manual(flat_idx, num_controls=2, treatment_shape=(2, 3)):
    if flat_idx < num_controls:
        return {'type': 'control', 'control': flat_idx, 'treatment': None, 'replicate': None}
    else:
        adjusted = flat_idx - num_controls
        t, r = np.unravel_index(adjusted, treatment_shape)
        return {'type': 'treatment', 'control': None, 'treatment': t, 'replicate': r}

for i in range(8):
    print(f"Sample {i}: {get_indices_manual(i)}")

Sample 0: {'type': 'control', 'control': 0, 'treatment': None, 'replicate': None}
Sample 1: {'type': 'control', 'control': 1, 'treatment': None, 'replicate': None}
Sample 2: {'type': 'treatment', 'control': None, 'treatment': np.int64(0), 'replicate': np.int64(0)}
Sample 3: {'type': 'treatment', 'control': None, 'treatment': np.int64(0), 'replicate': np.int64(1)}
Sample 4: {'type': 'treatment', 'control': None, 'treatment': np.int64(0), 'replicate': np.int64(2)}
Sample 5: {'type': 'treatment', 'control': None, 'treatment': np.int64(1), 'replicate': np.int64(0)}
Sample 6: {'type': 'treatment', 'control': None, 'treatment': np.int64(1), 'replicate': np.int64(1)}
Sample 7: {'type': 'treatment', 'control': None, 'treatment': np.int64(1), 'replicate': np.int64(2)}


**Limitation: No automatic state propagation.** You get indices as a tuple, but there's no connection to "counter" objects that track state.

### StateCounter Comparison

StateCounter handles complex structures declaratively:

In [9]:
from statecounter import Counter, Manager, product, stack

with Manager():
    control = Counter(num_states=2, name='control')
    treatment = Counter(num_states=2, name='treatment')
    replicate = Counter(num_states=3, name='replicate')
    
    # Complex structure: controls + (treatments × replicates)
    treatment_arm = product([treatment, replicate])
    samples = stack([control, treatment_arm])
    
    # Random access with automatic state propagation
    for i in [0, 1, 2, 5, 7]:
        samples.state = i
        print(f"Sample {i}: control={control.state}, treatment={treatment.state}, replicate={replicate.state}")

Sample 0: control=0, treatment=None, replicate=None
Sample 1: control=1, treatment=None, replicate=None
Sample 2: control=None, treatment=0, replicate=0
Sample 5: control=None, treatment=1, replicate=1
Sample 7: control=None, treatment=1, replicate=2


## Manual divmod Calculations

The most direct approach is to compute indices manually using `divmod`. This is covered in detail in the [Motivation](motivation.ipynb) page, but here's a quick example:

In [10]:
def get_product_indices(flat_idx, sizes):
    """Convert flat index to component indices for a Cartesian product."""
    indices = []
    for size in sizes:
        flat_idx, idx = divmod(flat_idx, size)
        indices.append(idx)
    return tuple(indices)

# 2 treatments × 3 replicates × 4 conditions
sizes = [2, 3, 4]
for flat in range(24):
    t, r, c = get_product_indices(flat, sizes)
    if flat < 5 or flat > 21:
        print(f"Sample {flat}: treatment={t}, replicate={r}, condition={c}")
    elif flat == 5:
        print("...")

Sample 0: treatment=0, replicate=0, condition=0
Sample 1: treatment=1, replicate=0, condition=0
Sample 2: treatment=0, replicate=1, condition=0
Sample 3: treatment=1, replicate=1, condition=0
Sample 4: treatment=0, replicate=2, condition=0
...
Sample 22: treatment=0, replicate=2, condition=3
Sample 23: treatment=1, replicate=2, condition=3


This works but has the same limitations discussed in the Motivation page:

1. **Doesn't compose** - Every new operation (stack, slice, shuffle) requires new math
2. **Error-prone** - Easy to make off-by-one errors or use wrong divisors
3. **No state tracking** - Returns tuples, not connected counter objects

## more-itertools

The [more-itertools](https://more-itertools.readthedocs.io/) library extends `itertools` with additional utilities, including some for random access:

In [11]:
# Note: You may need to install more-itertools: pip install more-itertools
try:
    from more_itertools import nth
    from itertools import product
    
    # Get the 4th element of a product
    treatments = range(2)
    replicates = range(3)
    
    # This still iterates through elements 0-3 internally
    sample_4 = nth(product(treatments, replicates), 4)
    print(f"Sample 4: {sample_4}")
except ImportError:
    print("more-itertools not installed - this example requires: pip install more-itertools")

more-itertools not installed - this example requires: pip install more-itertools


While `more-itertools` provides convenient functions, they still suffer from the fundamental limitation that iterators don't support O(1) random access. The `nth` function must consume elements up to index n.

## Comparison Summary

| Feature | itertools | NumPy | Manual divmod | more-itertools | StateCounter |
|---------|-----------|-------|---------------|----------------|--------------|
| **Random access** | No (O(n)) | Yes (O(1)) | Yes (O(1)) | No (O(n)) | Yes (O(1)) |
| **State propagation** | No | No | No | No | **Yes** |
| **Composable operations** | Limited | Array shapes only | Manual | Limited | **Yes** |
| **Stack (disjoint union)** | `chain` (no tracking) | Not supported | Manual | `chain` variants | **Built-in** |
| **Shuffle without materializing** | No | Array shuffle | Manual permutation | No | **Built-in** |
| **Sample/split** | Manual | Array slicing | Manual | Some utilities | **Built-in** |
| **Conflict detection** | N/A | N/A | Manual | N/A | **Automatic** |

## When to Use Each Approach

### Use itertools when:
- You just need to iterate through all combinations once
- Memory efficiency is critical and you don't need random access
- Your combinatorial space is simple (just products or chains)

### Use NumPy when:
- You're working with actual array data, not just indices
- Your structure is a regular multi-dimensional grid
- You need vectorized operations on the data

### Use manual divmod when:
- You have a one-off simple product
- Performance is critical and you want minimal overhead
- You're comfortable with the index math

### Use StateCounter when:
- You need random access to combinatorial spaces
- You want to track which component indices correspond to each element
- You need to compose operations (products, stacks, slices, shuffles)
- You want to shuffle, sample, or split without reimplementing index math
- You want automatic conflict detection for complex counter DAGs

## What StateCounter Intentionally Omits

StateCounter is designed for a specific purpose and intentionally doesn't try to replace other tools:

### Memory-efficient lazy iteration

`itertools` excels at lazy evaluation—it generates combinations on-the-fly without storing them all. StateCounter takes a different approach: it stores the *structure* of the combinatorial space (the counter DAG) and computes indices on demand. This enables O(1) random access but means StateCounter isn't optimized for cases where you truly need lazy iteration over massive spaces without random access.

### Vectorized array operations

NumPy is designed for fast numerical operations on arrays. StateCounter doesn't store or operate on array data—it manages *indices* into combinatorial spaces. If you need to perform math on the actual data at each combination, use NumPy for the computation and StateCounter for the index management.

### General-purpose data structures

StateCounter's counters are specialized for combinatorial enumeration. They're not meant to replace Python lists, NumPy arrays, or pandas DataFrames for general data storage.

## Combining StateCounter with Other Tools

StateCounter works well alongside other tools. Here's an example combining StateCounter with NumPy:

In [12]:
import numpy as np
from statecounter import Counter, Manager, product, shuffle, split

with Manager():
    # Define the combinatorial structure with StateCounter
    treatment = Counter(num_states=3, name='treatment')
    replicate = Counter(num_states=4, name='replicate')
    samples = product([treatment, replicate])
    
    # Shuffle and split using StateCounter
    shuffled = shuffle(samples, seed=42)
    train, test = split(shuffled, [0.8, 0.2])
    
    # Use NumPy for the actual data
    data = np.random.randn(3, 4)  # Some data indexed by (treatment, replicate)
    
    print("Test set samples:")
    test_values = []
    for _ in test:
        # StateCounter gives us the indices
        t, r = treatment.state, replicate.state
        # NumPy gives us the data
        value = data[t, r]
        test_values.append(value)
        print(f"  treatment={t}, replicate={r}, value={value:.3f}")
    
    print(f"\nTest set mean: {np.mean(test_values):.3f}")

Test set samples:
  treatment=1, replicate=0, value=-0.285
  treatment=1, replicate=3, value=-0.428

Test set mean: -0.357


## Summary

StateCounter fills a specific niche: **composable random access to combinatorial spaces with automatic state propagation**. It doesn't replace `itertools` for lazy iteration, NumPy for array operations, or simple `divmod` for one-off calculations. Instead, it provides a clean abstraction when you need to:

1. Access any point in a combinatorial space by index
2. Know the component indices for that point
3. Build complex structures from simpler ones
4. Shuffle, sample, split, or slice without reimplementing index math

For more details on StateCounter's capabilities, see:
- [Quick Start](quickstart.ipynb) - Get started with basic usage
- [Core Concepts](concepts.ipynb) - Understand the counter DAG and state propagation
- [Operations](operations.ipynb) - Complete reference for all operations